In [3]:
import pandas as pd
import numpy as np

!pip install openpyxl

# chnage from scientific notation 
pd.set_option('display.float_format', lambda x: '%.5f' % x)

data = pd.read_csv("../output/pur_data_full.csv")
                   

In [5]:
data.head()

,cooalpha,codalpha,comcode,statreg,eligibility,use,perref,statvalue,netmass,suppunit
0,AD,AD,84717098,1,e1,u10,202201,11623,8,48.00000
1,AD,AD,85235200,1,e1,u10,202201,21279,2,7.00000
2,AD,AD,85309000,1,e1,u10,202201,21030,370,0.00000
3,AD,ES,85235200,1,e1,u10,202201,152149,807,146821.00000
4,AD,FR,85423310,1,e1,u10,202201,1233,1,0.00000


In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 51000 entries, 0 to 50999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cooalpha     51000 non-null  str    
 1   codalpha     50994 non-null  str    
 2   comcode      51000 non-null  int64  
 3   statreg      51000 non-null  int64  
 4   eligibility  51000 non-null  str    
 5   use          51000 non-null  str    
 6   perref       51000 non-null  int64  
 7   statvalue    51000 non-null  int64  
 8   netmass      51000 non-null  int64  
 9   suppunit     51000 non-null  float64
dtypes: float64(1), int64(5), str(4)
memory usage: 3.9 MB


In [7]:
# filter for NAs in codalpha

data.isnull().sum()


cooalpha       0
codalpha       6
comcode        0
statreg        0
eligibility    0
use            0
perref         0
statvalue      0
netmass        0
suppunit       0
dtype: int64

In [9]:
na_df = data[data['codalpha'].isnull()]
na_df

,cooalpha,codalpha,comcode,statreg,eligibility,use,perref,statvalue,netmass,suppunit
30629,AE,NaN,73090090,1,e1,u10,202407,76753,11425,0.00000
30630,AE,NaN,84213985,1,e1,u10,202407,44768,6664,0.00000
36525,AE,NaN,84314300,1,e1,u10,202501,182368,3600,0.00000
37695,AG,NaN,84714100,1,e1,u10,202502,900,4,1.00000
40584,AE,NaN,84798997,1,e1,u10,202505,34794,2521,0.00000
48944,AO,NaN,84799070,1,e1,u10,202601,2196,64,0.00000


NA data is fine, can ignore. 

In [11]:
data["imports_ex_special"] = np.where(data["statreg"] == 1, data["statvalue"], 0)
data

,cooalpha,codalpha,comcode,statreg,eligibility,use,perref,statvalue,netmass,suppunit,imports_ex_special
0,AD,AD,84717098,1,e1,u10,202201,11623,8,48.00000,11623
1,AD,AD,85235200,1,e1,u10,202201,21279,2,7.00000,21279
2,AD,AD,85309000,1,e1,u10,202201,21030,370,0.00000,21030
3,AD,ES,85235200,1,e1,u10,202201,152149,807,146821.00000,152149
4,AD,FR,85423310,1,e1,u10,202201,1233,1,0.00000,1233
...,...,...,...,...,...,...,...,...,...,...,...
50995,AR,BE,10063067,1,e1,u11,202603,90353,80112,0.00000,90353
50996,AR,BE,10063098,1,e1,u11,202603,230979,236140,0.00000,230979
50997,AR,BE,38089327,1,e1,u11,202603,74049,13604,0.00000,74049
50998,AR,BE,39022000,1,e1,u11,202603,41961,23380,0.00000,41961


In [12]:
# full case logic for PUR data fields:

data["imports_ex_special"] = np.where(data["statreg"] == 1, data["statvalue"], 0)

data["eligibility_mfn"] = np.where(
    (data["eligibility"] == "e1") & (data["statreg"] == 1),
    data["statvalue"], 0
)

data["eligibility_gsp"] = np.where(
    (data["eligibility"] == "e2") & (data["statreg"] == 1) & (~data["use"].isin(["u10", "uzz"])),
    data["statvalue"], 0
)

data["eligibility_fta"] = np.where(
    (data["eligibility"] == "e3") & (data["statreg"] == 1) & (~data["use"].isin(["u10", "uzz"])),
    data["statvalue"], 0
)

data["eligibility_combined_pref"] = np.where(
    (data["eligibility"] == "e5") & (data["statreg"] == 1) & (~data["use"].isin(["u10", "uzz"])),
    data["statvalue"], 0
)

data["eligibility_unknown"] = np.where(
    (data["eligibility"] == "ez") & (data["statreg"] == 1),
    data["statvalue"], 0
)

data["eligibility_pref_mfn_0"] = np.where(
    (data["eligibility"].isin(["e2", "e3", "e5"])) & (data["statreg"] == 1) & (data["use"] == "u10"),
    data["statvalue"], 0
)

data["eligibility_pref_unknown"] = np.where(
    (data["eligibility"].isin(["e2", "e3", "e5"])) & (data["statreg"] == 1) & (data["use"] == "uzz"),
    data["statvalue"], 0
)

data["use_mfn_0"] = np.where(
    (data["statreg"] == 1) & (data["use"] == "u10"),
    data["statvalue"], 0
)

data["use_mfn_non_0"] = np.where(
    (data["statreg"] == 1) & (data["use"] == "u11"),
    data["statvalue"], 0
)

data["use_gsp_0"] = np.where(
    (data["statreg"] == 1) & (data["use"] == "u20"),
    data["statvalue"], 0
)

data["use_gsp_non_0"] = np.where(
    (data["statreg"] == 1) & (data["use"] == "u21"),
    data["statvalue"], 0
)

data["use_fta_0"] = np.where(
    (data["statreg"] == 1) & (data["use"] == "u30"),
    data["statvalue"], 0
)

data["use_fta_non_0"] = np.where(
    (data["statreg"] == 1) & (data["use"] == "u31"),
    data["statvalue"], 0
)

data["use_unknown"] = np.where(
    (data["statreg"] == 1) & (data["use"] == "uzz"),
    data["statvalue"], 0
)

data["eligibility_pref"] = np.where(
    (data["eligibility"].isin(["e2", "e3", "e5"])) & (data["statreg"] == 1) & (~data["use"].isin(["u10", "uzz"])),
    data["statvalue"], 0
)

data["use_pref"] = np.where(
    (data["statreg"] == 1) & (data["use"].isin(["u20", "u21", "u30", "u31"])),
    data["statvalue"], 0
)

data

,cooalpha,codalpha,comcode,statreg,eligibility,use,perref,statvalue,netmass,suppunit,...,eligibility_pref_unknown,use_mfn_0,use_mfn_non_0,use_gsp_0,use_gsp_non_0,use_fta_0,use_fta_non_0,use_unknown,eligibility_pref,use_pref
0,AD,AD,84717098,1,e1,u10,202201,11623,8,48.00000,...,0,11623,0,0,0,0,0,0,0,0
1,AD,AD,85235200,1,e1,u10,202201,21279,2,7.00000,...,0,21279,0,0,0,0,0,0,0,0
2,AD,AD,85309000,1,e1,u10,202201,21030,370,0.00000,...,0,21030,0,0,0,0,0,0,0,0
3,AD,ES,85235200,1,e1,u10,202201,152149,807,146821.00000,...,0,152149,0,0,0,0,0,0,0,0
4,AD,FR,85423310,1,e1,u10,202201,1233,1,0.00000,...,0,1233,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50995,AR,BE,10063067,1,e1,u11,202603,90353,80112,0.00000,...,0,0,90353,0,0,0,0,0,0,0
50996,AR,BE,10063098,1,e1,u11,202603,230979,236140,0.00000,...,0,0,230979,0,0,0,0,0,0,0
50997,AR,BE,38089327,1,e1,u11,202603,74049,13604,0.00000,...,0,0,74049,0,0,0,0,0,0,0
50998,AR,BE,39022000,1,e1,u11,202603,41961,23380,0.00000,...,0,0,41961,0,0,0,0,0,0,0


In [ ]:
# check values created for new variables
data.sum(numeric_only=True)

comcode                     2978398392637.00000
statreg                             53335.00000
perref                        10320918000.00000
statvalue                     17221542030.00000
netmass                       20840879732.00000
suppunit                        738599799.00000
imports_ex_special            15609762225.00000
eligibility_mfn               15445583391.00000
eligibility_gsp                  27074481.00000
eligibility_fta                 136656458.00000
eligibility_combined_pref               0.00000
eligibility_unknown                310270.00000
eligibility_pref_mfn_0             123285.00000
eligibility_pref_unknown            14340.00000
use_mfn_0                     13510535804.00000
use_mfn_non_0                  2005749194.00000
use_gsp_0                        23533736.00000
use_gsp_non_0                           0.00000
use_fta_0                        69292581.00000
use_fta_non_0                           0.00000
use_unknown                        65091

data looks correct and values present, move on to final aggregations for annual values

In [ ]:
# create year/month col

data["perref"] = data["perref"].astype(str)
data["year"] = data["perref"].str[:4]
data["month"] = data["perref"].str[-2:]

,cooalpha,codalpha,comcode,statreg,eligibility,use,perref,statvalue,netmass,suppunit,...,use_mfn_non_0,use_gsp_0,use_gsp_non_0,use_fta_0,use_fta_non_0,use_unknown,eligibility_pref,use_pref,year,month
0,AD,AD,84717098,1,e1,u10,202201,11623,8,48.00000,...,0,0,0,0,0,0,0,0,2022,01
1,AD,AD,85235200,1,e1,u10,202201,21279,2,7.00000,...,0,0,0,0,0,0,0,0,2022,01
2,AD,AD,85309000,1,e1,u10,202201,21030,370,0.00000,...,0,0,0,0,0,0,0,0,2022,01
3,AD,ES,85235200,1,e1,u10,202201,152149,807,146821.00000,...,0,0,0,0,0,0,0,0,2022,01
4,AD,FR,85423310,1,e1,u10,202201,1233,1,0.00000,...,0,0,0,0,0,0,0,0,2022,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50995,AR,BE,10063067,1,e1,u11,202603,90353,80112,0.00000,...,90353,0,0,0,0,0,0,0,2026,03
50996,AR,BE,10063098,1,e1,u11,202603,230979,236140,0.00000,...,230979,0,0,0,0,0,0,0,2026,03
50997,AR,BE,38089327,1,e1,u11,202603,74049,13604,0.00000,...,74049,0,0,0,0,0,0,0,2026,03
50998,AR,BE,39022000,1,e1,u11,202603,41961,23380,0.00000,...,41961,0,0,0,0,0,0,0,2026,03


In [23]:
# group by calculations for year level data:

group_cols = [ "cooalpha", "codalpha","comcode", "year" ]



sum_cols = [
        "statvalue", "imports_ex_special", "eligibility_mfn", "eligibility_gsp",
        "eligibility_fta", "eligibility_combined_pref", "eligibility_unknown",
        "eligibility_pref_mfn_0", "eligibility_pref_unknown",
        "use_mfn_0", "use_mfn_non_0", "use_gsp_0", "use_gsp_non_0",
        "use_fta_0", "use_fta_non_0", "use_unknown",
        "eligibility_pref", "use_pref"
    ]

pur_data = (
    data.groupby(group_cols, dropna=False)[sum_cols]
    .sum()
    .reset_index()
    .rename(columns={"svalue": "imports_total"})
    )


pur_data


,cooalpha,codalpha,comcode,year,statvalue,imports_ex_special,eligibility_mfn,eligibility_gsp,eligibility_fta,eligibility_combined_pref,...,eligibility_pref_unknown,use_mfn_0,use_mfn_non_0,use_gsp_0,use_gsp_non_0,use_fta_0,use_fta_non_0,use_unknown,eligibility_pref,use_pref
0,AD,AD,15119099,2024,3450,3450,3450,0,0,0,...,0,0,3450,0,0,0,0,0,0,0
1,AD,AD,24022090,2022,3671,3671,3671,0,0,0,...,0,0,3671,0,0,0,0,0,0,0
2,AD,AD,39219090,2022,940,940,0,0,940,0,...,0,0,940,0,0,0,0,0,940,0
3,AD,AD,39269097,2022,887,887,0,0,887,0,...,0,0,887,0,0,0,0,0,887,0
4,AD,AD,40169997,2024,3834,3834,0,0,3834,0,...,0,0,3834,0,0,0,0,0,3834,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19277,AT,AT,20097919,2023,105540,105540,0,0,105540,0,...,0,0,0,0,0,105540,0,0,105540,105540
19278,AT,AT,20098919,2023,4878,4878,0,0,4878,0,...,0,0,0,0,0,4878,0,0,4878,4878
19279,AT,AT,20099059,2023,5417,5417,0,0,5417,0,...,0,0,0,0,0,5417,0,0,5417,5417
19280,AT,AT,21011100,2023,3923,3923,0,0,3923,0,...,0,0,0,0,0,3923,0,0,3923,3923


In [ ]:
# calculate PUR variable

pur_data["pur_pct"] = np.where(
        pur_data["eligibility_pref"] > 0,
        pur_data["use_pref"] / pur_data["eligibility_pref"],
        np.nan
    )


,cooalpha,codalpha,comcode,year,statvalue,imports_ex_special,eligibility_mfn,eligibility_gsp,eligibility_fta,eligibility_combined_pref,...,use_mfn_0,use_mfn_non_0,use_gsp_0,use_gsp_non_0,use_fta_0,use_fta_non_0,use_unknown,eligibility_pref,use_pref,pur_pct
0,AD,AD,15119099,2024,3450,3450,3450,0,0,0,...,0,3450,0,0,0,0,0,0,0,NaN
1,AD,AD,24022090,2022,3671,3671,3671,0,0,0,...,0,3671,0,0,0,0,0,0,0,NaN
2,AD,AD,39219090,2022,940,940,0,0,940,0,...,0,940,0,0,0,0,0,940,0,0.00000
3,AD,AD,39269097,2022,887,887,0,0,887,0,...,0,887,0,0,0,0,0,887,0,0.00000
4,AD,AD,40169997,2024,3834,3834,0,0,3834,0,...,0,3834,0,0,0,0,0,3834,0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19277,AT,AT,20097919,2023,105540,105540,0,0,105540,0,...,0,0,0,0,105540,0,0,105540,105540,1.00000
19278,AT,AT,20098919,2023,4878,4878,0,0,4878,0,...,0,0,0,0,4878,0,0,4878,4878,1.00000
19279,AT,AT,20099059,2023,5417,5417,0,0,5417,0,...,0,0,0,0,5417,0,0,5417,5417,1.00000
19280,AT,AT,21011100,2023,3923,3923,0,0,3923,0,...,0,0,0,0,3923,0,0,3923,3923,1.00000


In [27]:
# final col names and selection

final_data = pur_data.rename(columns={

        "cooalpha": "exporting_iso_code_coo",
        "cooname": "exporting_country_name_coo",
        "codalpha": "exporting_iso_code_cod",
        "codname": "exporting_country_name_cod",
        "commodity_code": "cn8_product_code",
        "description": "product_description",
        "imports_total": "total_imports",
        "imports_ex_special": "total_imports_excl_spec_reg",
        "eligibility_mfn": "mfn_imports",
        "eligibility_gsp": "gsp_imports",
        "eligibility_fta": "fta_imports",
        "eligibility_combined_pref": "fta_and_gsp_imports",
        "eligibility_unknown": "unknown_imports",
        "eligibility_pref_mfn_0": "pref_eligible_entering_mfn0",
        "eligibility_pref_unknown": "pref_eligible_entering_unknown",
        "use_mfn_0": "mfn_zero_use",
        "use_mfn_non_0": "mfn_non_zero_use",
        "use_gsp_0": "gsp_zero_use",
        "use_gsp_non_0": "gsp_non_zero_use",
        "use_fta_0": "fta_zero_use",
        "use_fta_non_0": "fta_non_zero_use",
        "use_unknown": "unknown_use",
        "eligibility_pref": "pref_imports_eligible",
        "use_pref": "pref_imports_use"
    })

    # Match final SQL output types
final_data["year"] = final_data["year"].astype(int)

final_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 19282 entries, 0 to 19281
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   exporting_iso_code_coo          19282 non-null  str    
 1   exporting_iso_code_cod          19276 non-null  str    
 2   comcode                         19282 non-null  int64  
 3   year                            19282 non-null  int64  
 4   statvalue                       19282 non-null  int64  
 5   total_imports_excl_spec_reg     19282 non-null  int64  
 6   mfn_imports                     19282 non-null  int64  
 7   gsp_imports                     19282 non-null  int64  
 8   fta_imports                     19282 non-null  int64  
 9   fta_and_gsp_imports             19282 non-null  int64  
 10  unknown_imports                 19282 non-null  int64  
 11  pref_eligible_entering_mfn0     19282 non-null  int64  
 12  pref_eligible_entering_unknown  19282 non-n